In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RESULTS_DIR = Path("../results/summary")

sns.set_theme(style="whitegrid")

PLOT_DPI = 300

from pathlib import Path

PLOTS_DIR = Path("plots")
PLOTS_DIR.mkdir(exist_ok=True)

In [30]:
classification_fingerprints_df = pd.read_csv(RESULTS_DIR / "classification_fingerprints.csv")
classification_gnn_df = pd.read_csv(RESULTS_DIR / "classification_gnn.csv")
classification_filtered_gnn_df = pd.read_csv(RESULTS_DIR / "classification_filtered_gnn.csv")

regression_fingerprints_df = pd.read_csv(RESULTS_DIR / "regression_fingerprints.csv")
regression_gnn_df = pd.read_csv(RESULTS_DIR / "regression_gnn.csv")
regression_filtered_gnn_df = pd.read_csv(RESULTS_DIR / "regression_filtered_gnn.csv")


In [31]:
def aggregate_results(df, metrics):

    return (
        df.groupby(
            ["target", "model", "size"]
        )[metrics]
        .mean()
        .reset_index()
    )

In [32]:
classification_metrics = [
    "accuracy",
    "f1",
    "roc_auc",
]

classification_datasets = {
    "Fingerprint": aggregate_results(
        classification_fingerprints_df,
        classification_metrics,
    ),
    "GNN": aggregate_results(
        classification_gnn_df,
        classification_metrics,
    ),
    "Filtered GNN": aggregate_results(
        classification_filtered_gnn_df,
        classification_metrics,
    ),
}

In [33]:
regression_metrics = [
    "rmse",
    "mae",
    "r2",
]

regression_datasets = {
    "Fingerprint": aggregate_results(
        regression_fingerprints_df,
        regression_metrics,
    ),
    "GNN": aggregate_results(
        regression_gnn_df,
        regression_metrics,
    ),
    "Filtered GNN": aggregate_results(
        regression_filtered_gnn_df,
        regression_metrics,
    ),
}

In [34]:
import matplotlib.pyplot as plt


def plot_classification_model(
    datasets,
    target,
    model,
    metrics,
    task
):
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(18, 5),
        sharex=False,
    )

    for ax, metric in zip(
        axes,
        metrics,
    ):

        all_sizes = set()

        for dataset_name, df in datasets.items():

            subset = (
                df[
                    (df["target"] == target)
                    & (df["model"] == model)
                ]
                .sort_values("size")
            )

            if subset.empty:
                continue

            all_sizes.update(
                subset["size"].unique()
            )

            ax.plot(
                subset["size"],
                subset[metric],
                marker="o",
                label=dataset_name,
            )

        ax.set_title(
            metric.upper()
        )

        ax.set_xlabel(
            "Training size"
        )

        ax.grid(alpha=0.3)

        ax.set_xticks(
            sorted(all_sizes)
        )

    axes[0].set_ylabel("Score")

    handles, labels = (
        axes[0]
        .get_legend_handles_labels()
    )

    fig.legend(
        handles,
        labels,
        loc="upper left",
        ncol=3,
    )

    fig.suptitle(
        f"{target} | {task}",
        fontsize=16,
    )

    plt.tight_layout()
    # plt.show()

    filename = (
        PLOTS_DIR
        / f"{task}_{target}_{model}.png"
    )

    plt.savefig(
        filename,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close()

In [35]:
targets = ["A2a", "BACE1", "TYK2"]
models = ["mlp", "xgboost", "randomforest", "svm"]
for target in targets:
    for model in models:
        plot_classification_model(
            classification_datasets,
            target,
            model,
            classification_metrics,
            "classification"
        )

In [36]:
for target in targets:
    for model in models:
        plot_classification_model(
            regression_datasets,
            target,
            model,
            regression_metrics,
            "regression"
        )